<a href="https://colab.research.google.com/github/DivyaDharshiniG14/Mental-Health-Chatbot/blob/main/mentalhealthchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyngrok

In [ ]:
!pip install -q transformers datasets torch streamlit pyngrok accelerate

In [ ]:
from datasets import load_dataset
import pandas as pd
dataset = load_dataset("nbertagnolli/counsel-chat")
print("Example from the dataset:")
print(dataset['train'][10])
df = dataset['train'].to_pandas()
print("\nDataset Info:")
df.info()

In [ ]:

from transformers import AutoTokenizer
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
def format_and_tokenize_batch(batch):
    formatted_texts = [
        f"User: {q}\nBot: {a}{tokenizer.eos_token}"
        for q, a in zip(batch['questionText'], batch['answerText'])
    ]
    return tokenizer(
        formatted_texts,
        truncation=True,
        max_length=128,
        padding="max_length"
    )
filtered_dataset = dataset['train'].filter(lambda example: example['questionText'] is not None and example['answerText'] is not None)
tokenized_dataset = filtered_dataset.map(
    format_and_tokenize_batch,
    batched=True,
    batch_size=1000
)

print("\nExample of tokenized data:")
print(tokenized_dataset[10]['input_ids'][:30])

In [ ]:

import os
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
os.environ["WANDB_DISABLED"] = "true"
model = AutoModelForCausalLM.from_pretrained(model_name)
output_dir = "./mental_health_chatbot_model"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=200,
    fp16=True,
    report_to="none",
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)
print("🚀 Starting the fine-tuning process...")
trainer.train()
print("✅ Fine-tuning complete!")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Model saved to {output_dir}")

In [ ]:

from transformers import pipeline
print("Loading emotion classification model...")
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)
print("✅ Emotion model loaded.")

def get_emotion(text):
    """Classifies the emotion of a given text."""
    try:
        result = emotion_classifier(text)
        return result[0][0]['label']
    except Exception as e:
        return "neutral"
CRISIS_KEYWORDS = [
    "kill myself", "suicide", "want to die", "end my life",
    "self-harm", "hopeless", "can't go on", "no reason to live"
]

def is_crisis(text):
    """Detects crisis intent based on keywords."""
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in CRISIS_KEYWORDS)
print("\n--- Testing Helper Functions ---")
test_text_sad = "I feel so alone and I don't know what to do."
test_text_crisis = "I want to end my life, I see no other way out."
test_text_happy = "I finally finished my project and I am so relieved!"

print(f"Text: '{test_text_sad}'")
print(f"Emotion -> {get_emotion(test_text_sad)}")
print(f"Is Crisis? -> {is_crisis(test_text_sad)}")
print("-" * 20)
print(f"Text: '{test_text_crisis}'")
print(f"Emotion -> {get_emotion(test_text_crisis)}")
print(f"Is Crisis? -> {is_crisis(test_text_crisis)}")
print("-" * 20)
print(f"Text: '{test_text_happy}'")
print(f"Emotion -> {get_emotion(test_text_happy)}")
print(f"Is Crisis? -> {is_crisis(test_text_happy)}")
print("-" * 20)

In [ ]:

%%writefile app.py

import streamlit as st
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from collections import deque
st.set_page_config(page_title="MindfulBot", layout="centered", initial_sidebar_state="auto")

@st.cache_resource
def load_models():
    """Load all models and tokenizers only once."""
    model_path = "./mental_health_chatbot_model"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)
    emotion_classifier = pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        top_k=1
    )
    return tokenizer, model, emotion_classifier

tokenizer, model, emotion_classifier = load_models()

CRISIS_KEYWORDS = [
    "kill myself", "suicide", "want to die", "end my life",
    "self-harm", "hopeless", "can't go on", "no reason to live"
]

CRISIS_RESPONSE = """
It sounds like you are in a great deal of pain. Please know that help is available and you don't have to go through this alone.
It's important to talk to someone who can support you right now. Please contact a crisis hotline immediately.

- **USA & Canada:** Call or text 988
- **India:** Call +91 9999666555 (Vandrevala Foundation)
- **United Kingdom:** Call 111
- **Global:** Visit [Befrienders Worldwide](https://www.befrienders.org/) to find a helpline in your country.

Please, reach out to them. They can help.
"""

def is_crisis(text):
    """Simple keyword-based crisis detection."""
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in CRISIS_KEYWORDS)

def get_emotion(text):
    """Get the dominant emotion from text."""
    try:
        return emotion_classifier(text)[0][0]['label']
    except Exception:
        return "neutral"

st.title("🧠 MindfulBot")
st.markdown("Your AI companion for supportive conversations.")
st.markdown("🚨 **Disclaimer:** This is an AI proof-of-concept and not a substitute for a professional therapist. If you are in crisis, please contact the appropriate hotline listed in the sidebar.")

if "history" not in st.session_state:
    st.session_state.history = []
if "memory" not in st.session_state:
    st.session_state.memory = deque(maxlen=4)

with st.sidebar:
    st.header("Settings")
    use_empathetic_response = st.toggle("Enable Empathetic Phrases", value=True)
    if st.button("Clear Conversation History"):
        st.session_state.history = []
        st.session_state.memory.clear()
        st.rerun()
    st.divider()
    st.header("Emergency Hotlines")
    st.markdown(CRISIS_RESPONSE)

for message in st.session_state.history:
    with st.chat_message(message["role"]):
        if message["role"] == "user" and "emotion" in message:
            st.markdown(f"**Emotion:** `{message['emotion']}`\n\n{message['content']}")
        else:
            st.markdown(message["content"])

if prompt := st.chat_input("How are you feeling today?"):

    user_emotion = get_emotion(prompt)
    st.session_state.history.append({"role": "user", "content": prompt, "emotion": user_emotion})

    with st.chat_message("user"):
        st.markdown(f"**Emotion:** `{user_emotion}`\n\n{prompt}")

    if is_crisis(prompt):
        with st.chat_message("assistant"):
            st.warning("Crisis Intent Detected")
            st.markdown(CRISIS_RESPONSE)
        st.session_state.history.append({"role": "assistant", "content": CRISIS_RESPONSE})
    else:
        memory_prompt = "\n".join(st.session_state.memory) + f"\nUser: {prompt}\nBot:"

        inputs = tokenizer(memory_prompt, return_tensors="pt")

        with st.spinner("Thinking..."):
            output = model.generate(
                inputs["input_ids"],
                max_length=200,
                num_return_sequences=1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                top_k=40,
                top_p=0.95,
                no_repeat_ngram_size=3,
            )
        response = tokenizer.decode(output[0], skip_special_tokens=True)

        bot_response = response.split("Bot:")[-1].strip()
        empathetic_phrase = ""
        if use_empathetic_response and user_emotion in ["sadness", "fear", "anger", "disgust"]:
             empathetic_phrase = f"I hear that you're feeling {user_emotion}. "

        final_response = empathetic_phrase + bot_response

        with st.chat_message("assistant"):
            st.markdown(final_response)
        st.session_state.history.append({"role": "assistant", "content": final_response})
        st.session_state.memory.append(f"User: {prompt}")
        st.session_state.memory.append(f"Bot: {final_response}")

In [ ]:

from pyngrok import ngrok
import os
ngrok.kill()

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "32pwVuytQ0ZjFUbJgIARzM7jO6S_jWEUwmd5m3h5U2KzKXL9"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8501)
print(f"✅ Your Streamlit app is live at: {public_url}")
!streamlit run app.py --server.port 8501